# Лабораторна робота №5 (Варіант 2)
## Мультимодальні моделі для генерації описів зображень: Експериментальний конвеєр

**Дисципліна:** Технології та системи нейронних мереж  
**Університет:** Національний університет «Львівська політехніка»  
**Тема:** Мультимодальний аналіз зображень та генерація описів за допомогою LLaVA (Variant 2)  

### Мета роботи
Реалізувати повністю виконуваний експериментальний конвеєр для дослідження мультимодальної моделі `llava-hf/llava-interleave-qwen-0.5b-hf` на чотирьох візуально різнорідних сценах. Дослідити вплив:
1. Формулювання текстового промпту (стилі `concise`, `descriptive`, `detailed`).
2. Температури стохастичного декодування ($0.0, 0.2, 0.7, 1.0$).
3. Бюджету вихідних токенів ($32, 64, 128, 256$ токенів).

Підтримуються два режими виконання:
- `LIVE`: повноцінний локальний GPU-інференс на базі `transformers` та PyTorch.
- `ANALYSIS_ONLY`: завантаження верифікованих історичних результатів з `outputs/` без навантаження на GPU.

In [1]:
import os
import sys
import time
import json
from pathlib import Path
from PIL import Image
import pandas as pd
import torch

# Незалежне визначення робочої директорії
def resolve_lab_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / 'inputs').exists() and (current / 'outputs').exists():
        return current
    if (current / 'lab5_multimodal_captioning').exists():
        return current / 'lab5_multimodal_captioning'
    for p in [current] + list(current.parents):
        cand = p / 'lab5_multimodal_captioning'
        if cand.exists():
            return cand
    return current

LAB_DIR = resolve_lab_dir()
INPUTS_DIR = LAB_DIR / 'inputs'
OUTPUTS_DIR = LAB_DIR / 'outputs'
CAPTIONS_DIR = OUTPUTS_DIR / 'captions'
METADATA_DIR = OUTPUTS_DIR / 'metadata'
TABLES_DIR = OUTPUTS_DIR / 'tables'

CAPTIONS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# КЕРУВАННЯ РЕЖИМОМ ВИКОНАННЯ
# 'ANALYSIS_ONLY' -> детерміноване завантаження збережених артефактів
# 'LIVE'          -> виконання реального інференсу LLaVA на локальному GPU
# ---------------------------------------------------------
MODE = 'ANALYSIS_ONLY'  # Змініть на 'LIVE' для повного перерахунку на GPU
MODEL_ID = 'llava-hf/llava-interleave-qwen-0.5b-hf'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print(f'Active Lab 5 Directory: {LAB_DIR}')
print(f'Execution Mode: {MODE}')
print(f'Compute Device: {DEVICE}')

Active Lab 5 Directory: C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\max\lab5_multimodal_captioning
Execution Mode: ANALYSIS_ONLY
Compute Device: cuda:0


### 1. Ініціалізація мультимодального конвеєра LLaVA

У режимі `LIVE` завантажуються ваги моделі `llava-hf/llava-interleave-qwen-0.5b-hf` та відповідний процесор. У режимі `ANALYSIS_ONLY` блокнот працює з верифікованими історичними артефактами без зайвого навантаження на пам'ять GPU.

In [2]:
processor = None
model = None

if MODE == 'LIVE':
    from transformers import AutoProcessor, LlavaForConditionalGeneration
    print(f'Loading LLaVA model {MODEL_ID} into {DEVICE} (FP16)...')
    dtype = torch.float16 if 'cuda' in DEVICE else torch.float32
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        low_cpu_mem_usage=True
    ).to(DEVICE)
    model.eval()
    print('LLaVA pipeline successfully initialized.')
else:
    print('ANALYSIS_ONLY mode: Live model loading skipped. Using verified stored outputs.')

ANALYSIS_ONLY mode: Live model loading skipped. Using verified stored outputs.


### 2. Уніфікована функція експериментального інференсу

Функція `run_or_load_experiment` реалізує подвійний режим: у режимі `LIVE` генерує опис за допомогою `model.generate()`, вимірює точний час інференсу та записує файли на диск; у режимі `ANALYSIS_ONLY` зчитує збережені артефакти.

In [3]:
def run_or_load_experiment(
    run_id: str,
    image_id: str,
    image_file: str,
    category: str,
    series: str,
    prompt_style: str,
    prompt_text: str,
    temperature: float = 0.0,
    max_new_tokens: int = 128
) -> dict:
    txt_file = CAPTIONS_DIR / f'{run_id}.txt'
    json_file = METADATA_DIR / f'{run_id}.json'
    img_path = INPUTS_DIR / image_file
    
    if MODE == 'LIVE' and model is not None and processor is not None:
        image = Image.open(img_path).convert('RGB')
        full_prompt = f'<|im_start|>user\n<image>\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n'
        inputs = processor(text=full_prompt, images=image, return_tensors='pt').to(DEVICE)
        
        t0 = time.time()
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=(temperature > 0.0),
                temperature=temperature if temperature > 0.0 else None,
                top_p=0.9 if temperature > 0.0 else None
            )
        elapsed = time.time() - t0
        
        input_len = inputs.input_ids.shape[1]
        caption = processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()
        gen_tokens = len(output_ids[0]) - input_len
        tokens_sec = round(gen_tokens / elapsed, 2) if elapsed > 0 else 0.0
        
        record = {
            'run_id': run_id,
            'image_id': image_id,
            'image_file': image_file,
            'category': category,
            'series': series,
            'prompt_style': prompt_style,
            'prompt_text': prompt_text,
            'temperature': temperature,
            'max_new_tokens': max_new_tokens,
            'generated_tokens': gen_tokens,
            'char_count': len(caption),
            'latency_sec': round(elapsed, 4),
            'tokens_per_sec': tokens_sec,
            'caption': caption,
            'model_id': MODEL_ID,
            'device': DEVICE
        }
        with open(txt_file, 'w', encoding='utf-8') as f:
            f.write(caption)
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        return record
    else:
        # ANALYSIS_ONLY mode: read stored verified records without mutating files
        if json_file.exists():
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            data['provenance'] = 'historical_verified_metadata'
            return data
        elif txt_file.exists():
            with open(txt_file, 'r', encoding='utf-8') as f:
                cap = f.read().strip()
            # Never fabricate latency or token throughput when only text is available
            return {
                'run_id': run_id,
                'image_id': image_id,
                'image_file': image_file,
                'category': category,
                'series': series,
                'prompt_style': prompt_style,
                'prompt_text': prompt_text,
                'temperature': temperature,
                'max_new_tokens': max_new_tokens,
                'generated_tokens': len(cap.split()),
                'char_count': len(cap),
                'latency_sec': None,  # NOT_MEASURED
                'tokens_per_sec': None,  # NOT_MEASURED
                'caption': cap,
                'model_id': MODEL_ID,
                'device': 'unspecified',
                'provenance': 'historical_text_only_unmeasured'
            }
        raise FileNotFoundError(f'Neither live model nor historical record found for {run_id}')


### 3. Серія A: Дослідження впливу формулювання промпту (Prompt Wording)

Порівнюємо три стилістичні інструкції на всіх 4 тестових зображеннях (12 експериментів):
- `concise`: лаконічний однореченнєвий підпис.
- `descriptive`: структурований параграф з описом ключових об'єктів і контексту.
- `detailed`: вичерпний детальний опис об'єктів, просторових зв'язків та освітлення.

In [4]:
prompt_styles = {
    'concise': 'Provide a concise, single-sentence caption describing this image.',
    'descriptive': 'Describe this image in a well-structured paragraph, highlighting key objects, colors, and setting.',
    'detailed': 'Provide an exhaustive, highly detailed description of all visible objects, spatial relationships, materials, and lighting in this image.'
}

images_meta = [
    ('img1_lighthouse', '01_coastal_lighthouse.png', 'Landscape / Maritime'),
    ('img2_cafe', '02_cafe_bicycle.png', 'Urban / Street Scene'),
    ('img3_library', '03_library_interior.png', 'Interior / Architecture'),
    ('img4_robotics', '04_modern_robotics_lab.png', 'Technology / Laboratory')
]

series_a_results = []
for img_id, img_file, cat in images_meta:
    for p_style, p_text in prompt_styles.items():
        run_id = f'{img_id}_prompt_{p_style}'
        rec = run_or_load_experiment(
            run_id=run_id,
            image_id=img_id,
            image_file=img_file,
            category=cat,
            series='prompt_wording',
            prompt_style=p_style,
            prompt_text=p_text,
            temperature=0.0,
            max_new_tokens=128
        )
        series_a_results.append(rec)

df_a = pd.DataFrame(series_a_results)
print('=== Зведена таблиця серії А (12 експериментів) ===')
display(df_a[['image_id', 'prompt_style', 'generated_tokens', 'char_count', 'latency_sec', 'caption']])

=== Зведена таблиця серії А (12 експериментів) ===


,image_id,prompt_style,generated_tokens,char_count,latency_sec,caption
0,img1_lighthouse,concise,11,46,2.1464,A lighthouse on a cliff overlooking the ocean.
1,img1_lighthouse,descriptive,128,612,7.5330,The image presents a striking scene of a light...
2,img1_lighthouse,detailed,128,596,6.9473,The image presents a striking scene of a light...
3,img2_cafe,concise,12,46,0.8316,A bicycle parked next to a table and a window.
4,img2_cafe,descriptive,128,613,7.2121,The image presents a vibrant scene set in a qu...
5,img2_cafe,detailed,128,575,6.7524,"The image presents a scene set in a quaint, ol..."
6,img3_library,concise,11,38,0.7645,A room with a book shelf and a window.
7,img3_library,descriptive,119,577,6.6789,"The image presents a scene of a library, chara..."
8,img3_library,detailed,128,609,7.0063,"The image presents a scene of a library, chara..."
9,img4_robotics,concise,12,44,0.9253,A robot with a large arm and a large screen.


### 4. Серія B: Дослідження впливу температури стохастичного декодування

Фіксуємо промпт описового стилю (`descriptive`) на зображенні маяка (`01_coastal_lighthouse.png`) та варіюємо температуру: $0.0$ (жадібне декодування), $0.2$, $0.7$ та $1.0$.

In [5]:
series_b_results = []
temperatures = [0.0, 0.2, 0.7, 1.0]
for temp in temperatures:
    temp_suffix = str(temp).replace('.', '_')
    run_id = f'img1_lighthouse_temp_{temp_suffix}'
    rec = run_or_load_experiment(
        run_id=run_id,
        image_id='img1_lighthouse',
        image_file='01_coastal_lighthouse.png',
        category='Landscape / Maritime',
        series='temperature',
        prompt_style='descriptive',
        prompt_text=prompt_styles['descriptive'],
        temperature=temp,
        max_new_tokens=128
    )
    series_b_results.append(rec)

df_b = pd.DataFrame(series_b_results)
print('=== Зведена таблиця серії B (температурний свип) ===')
display(df_b[['temperature', 'generated_tokens', 'char_count', 'latency_sec', 'caption']])

=== Зведена таблиця серії B (температурний свип) ===


,temperature,generated_tokens,char_count,latency_sec,caption
0,0.0,128,612,7.1654,The image presents a striking scene of a light...
1,0.2,128,625,7.0994,The image presents a striking scene of a light...
2,0.7,114,532,6.3740,The image presents a stylized representation o...
3,1.0,128,598,7.6287,The image presents a simplistic yet captivatin...


### 5. Серія C: Дослідження впливу бюджету токенів (max_new_tokens)

Фіксуємо детальне формулювання промпту (`detailed`) на складній сцені лабораторії робототехніки (`04_modern_robotics_lab.png`) та варіюємо бюджет: $32, 64, 128, 256$ токенів.

In [6]:
series_c_results = []
token_budgets = [32, 64, 128, 256]
for budget in token_budgets:
    run_id = f'img4_robotics_tokens_{budget}'
    rec = run_or_load_experiment(
        run_id=run_id,
        image_id='img4_robotics',
        image_file='04_modern_robotics_lab.png',
        category='Technology / Laboratory',
        series='token_budget',
        prompt_style='detailed',
        prompt_text=prompt_styles['detailed'],
        temperature=0.0,
        max_new_tokens=budget
    )
    series_c_results.append(rec)

df_c = pd.DataFrame(series_c_results)
print('=== Зведена таблиця серії C (бюджет токенів) ===')
display(df_c[['max_new_tokens', 'generated_tokens', 'char_count', 'latency_sec', 'caption']])

=== Зведена таблиця серії C (бюджет токенів) ===


,max_new_tokens,generated_tokens,char_count,latency_sec,caption
0,32,32,137,1.9737,The image presents a scene set in a dimly lit ...
1,64,64,268,3.6193,The image presents a scene set in a dimly lit ...
2,128,128,582,7.6903,The image presents a scene set in a dimly lit ...
3,256,208,999,11.0402,The image presents a scene set in a dimly lit ...


### 6. Експорт підсумкового реєстру 20 експериментів

Об'єднуємо результати всіх 20 експериментів та зберігаємо у форматі CSV та JSON у папку `outputs/tables/`:

In [7]:
all_experiments = series_a_results + series_b_results + series_c_results
assert len(all_experiments) == 20, f'Expected 20 experiments, got {len(all_experiments)}'

df_all = pd.DataFrame(all_experiments)
csv_path = TABLES_DIR / 'caption_experiments.csv'
json_path = TABLES_DIR / 'caption_experiments.json'

df_all.to_csv(csv_path, index=False)
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_experiments, f, indent=2, ensure_ascii=False)

print(f'Збережено реєстр {len(df_all)} експериментів у {csv_path.name} та {json_path.name}')
print('Розподіл за серіями:')
print(df_all['series'].value_counts())

Збережено реєстр 20 експериментів у caption_experiments.csv та caption_experiments.json
Розподіл за серіями:
series
prompt_wording    12
temperature        4
token_budget       4
Name: count, dtype: int64


### 7. Висновки до блокноту 2

1. Реалізовано повністю автономний і повторюваний конвеєр експериментів із підтримкою режимів `LIVE` та `ANALYSIS_ONLY`.
2. Успішно досліджено всі 20 експериментів Варіанту 2 (12 за формулюванням промпту, 4 за температурою, 4 за бюджетом токенів).
3. Усі 20 підписів збережено у `outputs/captions/`, метадані зафіксовано в `outputs/metadata/`, а зведений реєстр експортовано в `outputs/tables/`.